## Artificial and Computational Intelligence Assignment 2

## Gaming with Min-Max Algorithm - Solution template

### List only the BITS (Name) of active contributors in this assignment:
1. KRISHNA KUMAR R(2021sc04040)
2. RAMESHWAR G(2021sc04894)
3. TRIPATHI KUSHALKUMAR SURESH(2021sc04896)
4. AJAY SAXENA(2021sc04160)

# Things to follow

1. Use appropriate data structures to represent the game using python libraries
2. Provide proper documentation
3. Create neat solution without error during game playing
4. REFER TO THE ASSIGNMENT INSTRUCTION & QUESTION INSTRUCTIONS OVER CANVAS FOR EVALUATION CRITERIA

### Coding begins here

In [1]:
#Code Block
# Define PLAYED_WORDS
PLAYED_WORDS = []

# Define a function to calculate the score of a word
def calculate_score(word):
    return len(word)

def min_max(current_player, depth):
    # Check if the game has ended
    game_ended = True
    for row in grid:
        if " " in row:
            game_ended = False
            break
    if game_ended:
        scores = {"player1": 0, "player2": 0}
        for row in grid:
            for letter in row:
                if letter.isalpha():
                    scores["player1" if letter.isupper() else "player2"] += 1
        return scores

    # Get the possible moves for the current player
    possible_moves = []
    for row in range(len(grid)):
        for col in range(len(grid[row])):
            if grid[row][col] == " ":
                for direction in ["across", "down"]:
                    for i in range(len(table)):
                        for j in range(len(table[i])):
                            if table[i][j] != " ":
                                if direction == "across":
                                    if col + len(table[i]) <= BOARD_SIZE and table[i][j] == grid[row][col+j]:
                                        possible_moves.append((table[i], row, col, direction))
                                else:
                                    if row + len(table[i]) <= BOARD_SIZE and table[i][j] == grid[row+i][col]:
                                        possible_moves.append((table[i], row, col, direction))

    # Evaluate each possible move
    best_score = float("-inf") if current_player == "player1" else float("inf")
    best_move = None
    for move in possible_moves:
        word, row, col, direction = move
        is_valid, word_score, _ = validate_move(word, row, col, direction)
        if not is_valid:
            continue
        score = calculate_score(word) * word_score
        PLAYED_WORDS.append(word)
        place_word(word, row, col, direction)
        if current_player == "player1":
            result = min_max("player2", depth-1)
            if result["player2"] - result["player1"] > best_score:
                best_score = result["player2"] - result["player1"]
                best_move = move
        else:
            result = min_max("player1", depth-1)
            if result["player1"] - result["player2"] < best_score:
                best_score = result["player1"] - result["player2"]
                best_move = move
        PLAYED_WORDS.pop()
        place_word(" " + word, row, col, direction)
    return {"move": best_move, "score": best_score}


### Terminal State / Game Ending (Win/Loss/Draw):  Implementation

In [2]:
# code Block

#Checks if the game is over
def is_game_over():
    for row in range(BOARD_SIZE):
        for col in range(BOARD_SIZE):
            if (row, col) not in PLAYED_LETTERS:
                return False
    return True

#Example to define a function for delaring the winner because it is the part of main function
def get_winner():
    scores = {"player1": 0, "player2": 0}
    for row in grid:
        for letter in row:
            if letter.isalpha():
                scores["player1" if letter.isupper() else "player2"] += 1
    if scores["player1"] > scores["player2"]:
        return "player1"
    elif scores["player2"] > scores["player1"]:
        return "player2"
    else:
        return "draw"


### Proper Prompt to the User for Dynamic Input & Handling incorrect inputs - Implement

In [3]:
# Define a function to get the player's move
def get_move(player_name):
    while True:
        try:
            word = input(f"{player_name}, enter your word: ")
            if not word.isalpha():
                raise ValueError("Word should only contain alphabets.")
            row = int(input("Enter the row number (0-14): "))
            if row < 0 or row > 14:
                raise ValueError("Row number should be between 0 and 14.")
            col = int(input("Enter the column number (0-14): "))
            if col < 0 or col > 14:
                raise ValueError("Column number should be between 0 and 14.")
            direction = input("Enter the direction (across or down): ")
            if direction != "across" and direction != "down":
                raise ValueError("Direction should be either 'across' or 'down'.")
            return word, row, col, direction
        except ValueError as e:
            print(f"Invalid input: {e}\nPlease try again.\n")



This above function uses a while loop and a try-except block to handle incorrect inputs from the user. It prompts the user for input, checks if the input is valid, and raises a ValueError if it is not. It keeps prompting the user until a valid input is provided. If an exception is raised, it catches the exception and prints an error message along with the exception message.

### Implementation of the Min-Max algorithm

In [4]:
#Code Block

def min_max(current_player, depth):
    # Check if the game has ended
    game_ended = True
    for row in grid:
        if " " in row:
            game_ended = False
            break
    if game_ended:
        scores = {"player1": 0, "player2": 0}
        for row in grid:
            for letter in row:
                if letter.isalpha():
                    scores["player1" if letter.isupper() else "player2"] += 1
        return scores

    # Get the possible moves for the current player
    possible_moves = []
    for row in range(len(grid)):
        for col in range(len(grid[0])):
            if grid[row][col] == " ":
                for word in WORDS:
                    for direction in ["across", "down"]:
                        valid, score, _ = validate_move(word, row, col, direction)
                        if valid:
                            possible_moves.append((score, (word, row, col, direction)))

    # Sort the possible moves by score
    possible_moves.sort(reverse=True)

    # Get the best possible move
    if current_player == "player1":
        best_move = None
        best_score = float("-inf")
        for move in possible_moves:
            score, move_details = move
            _, _, _, direction = move_details
            next_player = "player2"
            if direction == "down":
                next_player = current_player
            place_word(*move_details)
            PLAYED_WORDS.append(move_details[0])
            score = min_max(next_player, depth+1)["player1"]
            undo_move(move_details)
            PLAYED_WORDS.pop()
            if score > best_score:
                best_score = score
                best_move = move_details
    else:
        best_move = None
        best_score = float("inf")
        for move in possible_moves:
            score, move_details = move
            _, _, _, direction = move_details
            next_player = "player1"
            if direction == "down":
                next_player = current_player
            place_word(*move_details)
            PLAYED_WORDS.append(move_details[0])
            score = min_max(next_player, depth+1)["player2"]
            undo_move(move_details)
            PLAYED_WORDS.pop()
            if score < best_score:
                best_score = score
                best_move = move_details

    return {"player1": best_score, "player2": -best_score}[current_player] if depth == 0 else {"player1": best_score, "player2": best_score}[current_player]


To implement the Two-player Crossword Puzzle game using Min-Max algorithm in Python with dynamic inputs, we can follow the following steps:

1.	Define the game board with a 2D array representing the grid and another 2D array representing the table of words.<br>
2.	Define a function to display the game board.<br>
3.	Define a function to get the player's move, i.e., the word to be placed and its position.<br>
4.	Define a function to validate the player's move.<br>
5.	Define a function to check if the game has ended.<br>
6.	Define a function to evaluate the game board, i.e., calculate the score for each player based on the character count in the words placed on the board.<br>
7.	Define the Min-Max algorithm function to determine the best move for the computer player.<br>
8.	Define the main game function that takes input from both players and updates the game board.<br>

Here is the implementation of the Two-player Crossword Puzzle game using Min-Max algorithm in Python:

In [8]:

import copy

# Define the game board
grid = [
    ["1", " ", " ", " ", "#", "2", "#", "#"],
    ["#", "#", "#", "#", "4", " ", " ", "#"],
    ["#", "#", "#", "#", "#", " ", "#", "3"],
    ["5", "#", "7", "#", "#", " ", "#", " "],
    [" ", " ", " ", " ", " ", " ", " ", " "],
    [" ", "#", " ", "#", "#", " ", "#", "#"],
    ["#", "#", "#", "#", "8", " ", " ", "#"],
    ["6", " ", " ", "#", "#", " ", "#", "#"],
]

table = [
    ["c", "b", "k", "c", "e", "d", "r", "h"],
    ["r", "a", "a", "a", "l", "o", "a", "e"],
    ["o", "t", "n", "t", "e", "g", "t", "n"],
    ["w", " ", "g", " ", "p", " ", " ", " "],
    [" ", " ", "a", " ", "h", " ", " ", " "],
    [" ", " ", "r", " ", "a", " ", " ", " "],
    [" ", " ", "o", " ", "n", " ", " ", " "],
    [" ", " ", "o", " ", "t", " ", " ", " "],
]

#define the board size
BOARD_SIZE = 15

# Define a function to display the game board
def display_board():
    print("Grid:")
    for row in grid:
        print("|" + "|".join(row) + "|")
        print("-" * 26)
    print("\nTable:")
    for row in table:
        print("|" + "|".join(row) + "|")
        print("-" * 26)

# Define a function to get the player's move
def get_move(player_name):
    word = input(f"{player_name}, enter your word: ")
    row = int(input("Enter the row number (0-14): "))
    col = int(input("Enter the column number (0-14): "))
    direction = input("Enter the direction (across or down): ")
    return word, row, col, direction




# Define a function to validate the player's move
def validate_move(word, row, col, direction):
    # Check if the word fits on the board
    if direction == 'down':
        if row + len(word) > BOARD_SIZE:
            return False, 0, None
    else:
        if col + len(word) > BOARD_SIZE:
            return False, 0, None

    # Check if the word intersects with any existing words
    for i, letter in enumerate(word):
        if direction == 'down':
            if (row + i, col) in PLAYED_LETTERS:
                if PLAYED_LETTERS[(row + i, col)] != letter:
                    return False, 0, None
            elif (row + i, col) not in PLAYED_LETTERS:
                if col > 0 and (row + i, col - 1) in PLAYED_LETTERS:
                    return False, 0, None
                if col < BOARD_SIZE - 1 and (row + i, col + 1) in PLAYED_LETTERS:
                    return False, 0, None
        else:
            if (row, col + i) in PLAYED_LETTERS:
                if PLAYED_LETTERS[(row, col + i)] != letter:
                    return False, 0, None
            elif (row, col + i) not in PLAYED_LETTERS:
                if row > 0 and (row - 1, col + i) in PLAYED_LETTERS:
                    return False, 0, None
                if row < BOARD_SIZE - 1 and (row + 1, col + i) in PLAYED_LETTERS:
                    return False, 0, None

    return True, len(word), direction


# Define a function to place the player's move on the game board
Grid = [[" " for i in range(BOARD_SIZE)] for j in range(BOARD_SIZE)]


# Define PLAYED_WORDS at the beginning of your script
PLAYED_WORDS = []

#def place_word(word, row, col, direction):     
def place_word(word, row, col, direction):
    global grid
    for i, letter in enumerate(word):
        if direction == "across":
            grid[row][col+i] = letter
        else:
            grid[row+i][col] = letter





#Define a function to calculate the score of a word
def calculate_score(word):
    return len(word)

def min_max(current_player, depth):
    # Check if the game has ended
    game_ended = True
    for row in grid:
        if " " in row:
            game_ended = False
            break
    if game_ended:
        scores = {"player1": 0, "player2": 0}
        for row in grid:
            for letter in row:
                if letter.isalpha():
                    scores["player1" if letter.isupper() else "player2"] += 1
        return scores

    # Get the possible moves for the current player
    possible_moves = []
    for row in range(len(grid)):
        for col in range(len(grid[0])):
            if grid[row][col] == " ":
                for word in table:
                    for direction in ["horizontal", "vertical"]:
                        if validate_move(word, row, col, direction):
                            possible_moves.append((word, row, col, direction))

    # Apply Min-Max algorithm recursively
    if current_player == "player1":
        best_score = float('-inf')
        for move in possible_moves:
            word, row, col, direction = move
            new_grid = place_word(word, row, col, direction)
            score = calculate_score(word) + min_max("player2", depth+1)["player2"]
            if score > best_score:
                best_score = score
                if depth == 0:
                    best_move = move
        if depth == 0:
            return best_move
        else:
            return {"player1": best_score}
    elif current_player == "player2":
        best_score = float('inf')
        for move in possible_moves:
            word, row, col, direction = move
            new_grid = place_word(word, row, col, direction)
            score = calculate_score(word) + min_max("player1", depth+1)["player1"]
            if score < best_score:
                best_score = score
        return {"player2": best_score}

# Define global variables
PLAYED_LETTERS = {}

#Checks if the game is over
def is_game_over():
    for row in range(BOARD_SIZE):
        for col in range(BOARD_SIZE):
            if (row, col) not in PLAYED_LETTERS:
                return False
    return True

    
#Define the main function
def main():
    print("Welcome to the Two-Player Crossword Puzzle Game!\n")
    display_board()

    # Initialize variables
    player1_score = 0
    player2_score = 0
    current_player = 1

    # Start the game loop
    while True:
        # Get the player's move
        #word, row, col = get_move(f"Player {current_player}")
        word, row, col, direction = get_move(f"Player {current_player}")


        # Validate the player's move
        move_valid = validate_move(word, row, col, direction)
        if move_valid:
            valid_move, word_length, word_direction = move_valid
        else:
            print("Invalid move! Try again.\n")

            
            
        # Place the word on the grid
        place_word(word, row, col, word_direction)
        display_board()

        # Update the player's score
        if current_player == 1:
            player1_score += word_length
        else:
            player2_score += word_length

        # Check if the game is over
        if is_game_over():
            break

        # Switch the player
        current_player = 3 - current_player

    # End of the game
    print("Game Over!")
    print(f"Player 1 Score: {player1_score}")
    print(f"Player 2 Score: {player2_score}")
    if player1_score > player2_score:
        print("Player 1 Wins!")
    elif player2_score > player1_score:
        print("Player 2 Wins!")
    else:
        print("It's a tie!")


To start the game, we can call the main function **main()** that we have defined in the code. This will initialize the game and start the gameplay. The function will display the game board, and then prompt the first player to make their move by entering the word to be placed and its starting position. The game will then validate the move and update the game board accordingly. The next player will then be prompted to make their move, and so on, until all the words have been correctly placed on the game board.



In [7]:
#Call the main function
main()